In [1]:
!pip install wfdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 9.5 MB/s eta 0:00:00


In [2]:
import os
import math
import torch
import torch.nn as nn
import numpy as np
from scipy.io import loadmat
from scipy.signal import resample
from torch.utils.data import Dataset, DataLoader, random_split
from torch.amp import autocast
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.notebook import tqdm

# ==========================================
# 1. FIXED TRANSFORMER ARCHITECTURE
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class AdvancedECGTransformer(nn.Module):
    def __init__(self, in_channels=12, d_model=256, nhead=8, num_classes=5, num_layers=6):
        super().__init__()
        self.d_model = d_model
        
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, d_model // 2, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(d_model // 2),
            nn.GELU(),
            nn.Conv1d(d_model // 2, d_model, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(d_model),
            nn.GELU()
        )
        self.pos_encoder = PositionalEncoding(d_model=d_model)
        
        # Critical deep-transformer fixes: norm_first=True
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4, 
            dropout=0.1, activation='gelu', batch_first=True,
            norm_first=True 
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers, norm=nn.LayerNorm(d_model)
        )
        
        self.head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, x):
        x = self.stem(x) 
        x = x.transpose(1, 2) 
        x = x * math.sqrt(self.d_model) # Prevent features from drowning in positional noise
        x = self.pos_encoder(x)
        x = self.transformer(x)
        x_avg = torch.mean(x, dim=1) # Global Average Pooling routes gradients to all tokens
        return self.head(x_avg)

# ==========================================
# 2. SAFEGUARDED EVALUATION FUNCTION
# ==========================================
def evaluate_model_advanced(model, dataloader, device):
    model.eval()
    all_probs, all_targets = [], []
    
    with torch.no_grad():
        for signals, labels in dataloader:
            signals = signals.to(device, non_blocking=True)
            with autocast('cuda', dtype=torch.bfloat16):
                probs = torch.sigmoid(model(signals)) 
            all_probs.append(probs.cpu().float().numpy())
            all_targets.append(labels.numpy())
            
    all_probs = np.vstack(all_probs)
    all_targets = np.vstack(all_targets)
    
    aucs, auprcs = [], []
    superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
    
    for i in range(len(superclasses)):
        if len(np.unique(all_targets[:, i])) == 2:
            aucs.append(roc_auc_score(all_targets[:, i], all_probs[:, i]))
            auprcs.append(average_precision_score(all_targets[:, i], all_probs[:, i]))
            
    macro_auc = np.mean(aucs) if len(aucs) > 0 else 0.0
    macro_auprc = np.mean(auprcs) if len(auprcs) > 0 else 0.0
        
    return macro_auc, macro_auprc

In [6]:
# ==========================================
# 3. CPSC 2018 DATASET (DYNAMIC PADDING LOADER)
# ==========================================
class ChapmanDataset(Dataset):
    def __init__(self, data_path, target_hz=100, original_hz=500):
        super().__init__()
        self.data_path = data_path
        self.superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
        self.target_length = target_hz * 10 # Enforces exactly 1000 sequence length
        self.target_hz = target_hz
        self.original_hz = original_hz
        
        self.snomed_mapping = {
            '426783006': 'NORM', '426177001': 'NORM', '427084000': 'NORM', '427393009': 'NORM',
            '164865005': 'MI', '164861001': 'MI', '54329005': 'MI',
            '164934002': 'STTC', '59931005': 'STTC', '164917005': 'STTC', '111975006': 'STTC',
            '164909002': 'CD', '733534002': 'CD', '59118001': 'CD', '713427006': 'CD', 
            '713426002': 'CD', '445118002': 'CD', '270492004': 'CD', '164947007': 'CD', '6374002': 'CD',
            '164873001': 'HYP', '370355005': 'HYP'
        }

        self.files = [f[:-4] for f in os.listdir(data_path) if f.endswith('.mat') and not f.startswith('.')]
        
        self.signals = []
        self.labels = []
        self.skipped_files = 0
        
        print(f"Pre-loading CPSC 2018 | Resampling to {target_hz}Hz | Enforcing 10s Sequence...")
        for f in tqdm(self.files, desc="Processing ECGs"):
            mat_path = os.path.join(self.data_path, f + '.mat')
            hea_path = os.path.join(self.data_path, f + '.hea')
            
            try:
                # 1. Read Native MATLAB Signal
                mat_data = loadmat(mat_path)
                key = 'val' if 'val' in mat_data else [k for k in mat_data.keys() if not k.startswith('__')][0]
                signal = np.array(mat_data[key], dtype=np.float32)
                
                if signal.shape[0] == 12:
                    signal = signal.T 
                
                signal = signal / 1000.0 
                signal = np.nan_to_num(signal)
                
                # --- DYNAMIC TEMPORAL RESAMPLING ---
                time_ratio = self.target_hz / self.original_hz
                dynamic_samples = int(signal.shape[0] * time_ratio)
                signal = resample(signal, dynamic_samples, axis=0) 
                
                # --- ENFORCE 10-SECOND SEQUENCE ---
                if signal.shape[0] > self.target_length:
                    start = (signal.shape[0] - self.target_length) // 2
                    signal = signal[start : start + self.target_length, :]
                elif signal.shape[0] < self.target_length:
                    pad_total = self.target_length - signal.shape[0]
                    pad_top = pad_total // 2
                    pad_bottom = pad_total - pad_top
                    signal = np.pad(signal, ((pad_top, pad_bottom), (0, 0)), mode='constant')
                
                signal_tensor = torch.tensor(signal, dtype=torch.float32).transpose(0, 1)
                
                # 2. Extract Labels safely
                with open(hea_path, 'r', encoding='utf-8-sig', errors='ignore') as text_file:
                    lines = text_file.readlines()
                
                dx_str = ""
                for line in lines:
                    if line.startswith('#Dx:') or line.startswith('# Dx:'):
                        dx_str = line.split(':')[1].strip()
                        break
                
                if not dx_str:
                    self.skipped_files += 1
                    continue
                
                label = np.zeros(len(self.superclasses))
                for code in dx_str.split(','):
                    code = code.strip()
                    if code in self.snomed_mapping:
                        idx = self.superclasses.index(self.snomed_mapping[code])
                        label[idx] = 1
                        
                self.signals.append(signal_tensor)
                self.labels.append(torch.tensor(label, dtype=torch.float32))
                
            except Exception as e:
                self.skipped_files += 1
                continue

        print(f"Successfully loaded {len(self.signals)} records. Skipped {self.skipped_files} files.")

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, idx):
        return self.signals[idx], self.labels[idx]

In [7]:
# 1. Setup Directories
CHAPMAN_DIR = "/kaggle/input/datasets/erarayamorenzomuten/chapmanshaoxing-12lead-ecg-database/WFDB_ChapmanShaoxing"

# 2. Load and Split the Dataset (80% Train, 20% Validation)
full_dataset = ChapmanDataset(data_path=CHAPMAN_DIR, target_hz=100)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
generator = torch.Generator().manual_seed(42) # Ensures identical 1:1 split vs Mamba

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

# num_workers=0 remains critical to safeguard Kaggle RAM
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

print(f"Training on {train_size} records. Validating on {val_size} records.")

# 3. Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the SOTA Transformer baseline
model = AdvancedECGTransformer(
    in_channels=12, d_model=256, nhead=8, num_classes=5, num_layers=6
).to(device)

# Using the lowered 3e-4 Learning Rate for Transformer Stability
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
criterion = nn.BCEWithLogitsLoss()

# 4. The Training Loop
epochs = 30
max_norm = 1.0

print("Starting Transformer Baseline Training on Chapman-Shaoxing...")
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    valid_batches = 0
    
    for i, (signals, labels) in enumerate(train_dataloader):
        signals, labels = signals.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True) 
        
        with autocast('cuda', dtype=torch.bfloat16):
            outputs = model(signals)
            loss = criterion(outputs, labels)
            
        if torch.isnan(loss):
            continue
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
        
        optimizer.step()
        total_loss += loss.item()
        valid_batches += 1
        
    scheduler.step()
    avg_train_loss = total_loss / max(valid_batches, 1)
    
    # Live Validation check at the end of each epoch
    val_auc, val_auprc = evaluate_model_advanced(model, val_dataloader, device)
    
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val AUC: {val_auc:.4f} | Val AUPRC: {val_auprc:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

# Save the weights
torch.save(model.state_dict(), '/kaggle/working/transformer_chapman_weights.pth')
print("Transformer Model saved successfully!")

Pre-loading CPSC 2018 | Resampling to 100Hz | Enforcing 10s Sequence...


Processing ECGs:   0%|          | 0/10247 [00:00<?, ?it/s]

Successfully loaded 10247 records. Skipped 0 files.
Training on 8197 records. Validating on 2050 records.


/tmp/ipykernel_58/2759206016.py:51: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Starting Transformer Baseline Training on Chapman-Shaoxing...
Epoch [1/30] | Train Loss: 0.2410 | Val AUC: 0.8952 | Val AUPRC: 0.4889 | LR: 0.000293
Epoch [2/30] | Train Loss: 0.1495 | Val AUC: 0.9300 | Val AUPRC: 0.5054 | LR: 0.000271
Epoch [3/30] | Train Loss: 0.1184 | Val AUC: 0.9464 | Val AUPRC: 0.5238 | LR: 0.000238
Epoch [4/30] | Train Loss: 0.1054 | Val AUC: 0.9514 | Val AUPRC: 0.5536 | LR: 0.000197
Epoch [5/30] | Train Loss: 0.0973 | Val AUC: 0.9539 | Val AUPRC: 0.5821 | LR: 0.000150
Epoch [6/30] | Train Loss: 0.0893 | Val AUC: 0.9553 | Val AUPRC: 0.5847 | LR: 0.000104
Epoch [7/30] | Train Loss: 0.0806 | Val AUC: 0.9485 | Val AUPRC: 0.5823 | LR: 0.000063
Epoch [8/30] | Train Loss: 0.0718 | Val AUC: 0.9527 | Val AUPRC: 0.5871 | LR: 0.000030
Epoch [9/30] | Train Loss: 0.0663 | Val AUC: 0.9521 | Val AUPRC: 0.5867 | LR: 0.000008
Epoch [10/30] | Train Loss: 0.0614 | Val AUC: 0.9523 | Val AUPRC: 0.5871 | LR: 0.000300
Epoch [11/30] | Train Loss: 0.0917 | Val AUC: 0.9414 | Val AUPRC: 0